# 청년 복지정보 크롤러 (민간 카테고리)

## 개요
Selenium과 BeautifulSoup을 사용해 정부 복지정보 포털의 **민간** 카테고리에서 청년 대상 복지 서비스 정보를 자동으로 수집하는 크롤러입니다. 전체 페이지를 순회하며 각 항목의 상세 페이지에 진입해 기관명, 사업목적, 지원대상, 지원내용, 신청방법, 제출서류, 첨부파일 등 세부 필드를 추출한 뒤 CSV / TSV / JSON 세 가지 형식으로 저장합니다.

> **저작권 안내**: 이 코드가 접근하는 실제 사이트 도메인은 저작권 문제로 `target_site`라는 이름으로 치환하였습니다.

## 주요 기능
1. **라벨 매핑 (`LABEL_MAPPING`)** — 사업목적 / 지원대상 / 지원내용 / 신청방법 / 제출서류 / 첨부파일 등 항목명을 표준 필드명으로 통일합니다.
2. **동적 콘텐츠 대기 (`wait_for_dynamic_content`)** — JS로 렌더링되는 요소가 로드될 때까지 대기한 뒤 파싱을 시작합니다.
3. **정보 추출 함수**
   - `extract_basic_info` — 제목, 기관명, 사업상태, 사업기간, 연락처, 이메일 등 기본 정보 추출 (기관명은 최대 3단계 fallback으로 탐색, 연락처·이메일은 정규식 보조 추출)
   - `extract_section_content` — 사업목적 / 지원대상 / 지원내용 / 신청방법 / 제출서류 섹션별 본문 추출, 다른 섹션과 내용이 중복되는 경우 제외
   - `extract_attachments` — 첨부파일명 목록 추출
4. **`extract_detail_info`** — 상세 페이지 진입 후 기본 정보 + 섹션별 내용 + 첨부파일을 모두 모아 하나의 딕셔너리로 구성합니다.
5. **메인 루프** — 페이지(1~11) → 항목("자세히 보기" 버튼) 순서로 순회하며 `상세 페이지 방문 → 데이터 수집 → 목록으로 복귀`를 반복합니다.
6. **`save_data_to_files`** — 수집된 데이터를 pandas `DataFrame`으로 변환한 뒤 CSV / TSV / JSON 세 형식으로 저장합니다.

## 코드 구조
- **1. 유틸리티 및 매핑 설정** — 라벨 매핑, 텍스트 정제(`clean_text`)
- **2. 민간 복지정보 크롤링 로직** — 기본 정보 / 섹션 / 첨부파일 추출 함수들
- **3. 메인 실행 블록** — Selenium 드라이버 설정, 페이지/항목 반복, 데이터 저장

## 설계 포인트
- **다단계 fallback 탐색**: 기관명처럼 사이트 구조가 일정하지 않은 필드는 CSS 클래스 탐색 → XPath 재시도 → 구조적 패턴 매칭까지 최대 3단계로 시도합니다.
- **섹션 간 중복 제거**: 같은 텍스트가 여러 섹션에 겹쳐 추출되는 것을 막기 위해 `all_section_contents`로 이미 추출된 내용과 비교합니다.
- **테스트 모드 지원**: `TEST_MODE` / `COLLECT_ALL_ITEMS` 플래그로 첫 항목 1개만 빠르게 테스트하거나 전체 수집으로 전환할 수 있습니다.


In [ ]:
## 타겟사이트 청년 복지정보 크롤러 - 민간 전체 항목
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup
import pandas as pd
import time
import sys
import os 
import json 
import re

# ========================================
# 1. 유틸리티 및 매핑 설정
# ========================================

# 모든 섹션 레이블 정의 및 매핑
LABEL_MAPPING = {
    '사업목적': '사업목적',
    '지원대상': '지원대상', 
    '지원내용': '지원내용', 
    '지원 내용': '지원내용',
    '신청방법': '신청방법', 
    '제출서류': '제출서류',
    '첨부파일': '첨부파일'
}

# 탭 ID와 구분을 매핑
TAB_SECTIONS = {1: '중앙정부', 2: '지자체', 3: '민간'}

def clean_text(text):
    """텍스트 정리: 불필요한 공백/개행 제거 및 잡음 필터링"""
    if text is None: return ""
    text = text.replace("이 누리집은 대한민국 공식 전자정부 누리집입니다.", "")
    text = text.replace("[새창열림]링크 이동", "")
    text = text.replace("[새창열림]파일 미리보기", "")
    return ' '.join(text.split()).strip()

# ========================================
# 2. 민간 복지정보 크롤링 로직
# ========================================

def wait_for_dynamic_content(driver, timeout=5):
    """동적 콘텐츠 로딩 대기 (여러 방법 시도)"""
    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.cl-htmlsnippet, div.wlfare-info-nm, div.line-tit"))
        )
    except TimeoutException:
        pass
    
    time.sleep(2)
    driver.execute_script("return document.readyState")
    time.sleep(1)


def extract_basic_info(driver, debug=False):
    """기본 정보 추출 (제목, 기관명, 사업상태, 사업기간, 연락처, 이메일)"""
    basic_info = {}
    
    try:
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # 제목 추출
        try:
            title_element = driver.find_element(By.CSS_SELECTOR, "div.wlfare-info-nm div.cl-text")
            basic_info['제목'] = clean_text(title_element.text)
        except:
            basic_info['제목'] = '제목 없음'
        
        # 기관명 추출 - 개선된 버전
        try:
            # 방법 1: diamond-blt 클래스로 찾기
            org_found = False
            
            # diamond-blt 클래스를 가진 div 찾기
            diamond_elements = soup.find_all('div', class_='diamond-blt')
            for elem in diamond_elements:
                elem_text = clean_text(elem.get_text())
                if '기관명' in elem_text:
                    # 부모의 부모 찾기
                    parent = elem.find_parent('div')
                    if parent:
                        grandparent = parent.find_parent('div')
                        if grandparent:
                            # 형제 요소들 중에서 값 찾기
                            siblings = grandparent.find_all('div', class_='cl-text')
                            for sibling in siblings:
                                text = clean_text(sibling.get_text())
                                if text and text != '기관명' and len(text) > 2:
                                    basic_info['기관명'] = text
                                    org_found = True
                                    if debug:
                                        print(f"         ✓ 기관명 추출 (방법1): {text}")
                                    break
                    if org_found:
                        break
            
            # 방법 2: XPath로 재시도
            if not org_found:
                try:
                    org_xpath = "//div[contains(text(), '기관명')]"
                    org_elements = driver.find_elements(By.XPATH, org_xpath)
                    for org_elem in org_elements:
                        try:
                            # 부모 컨테이너 찾기
                            parent = org_elem.find_element(By.XPATH, "../..")
                            # 값을 가진 div 찾기
                            value_divs = parent.find_elements(By.CSS_SELECTOR, "div.cl-text")
                            for value_div in value_divs:
                                text = clean_text(value_div.text)
                                if text and text != '기관명' and len(text) > 2:
                                    basic_info['기관명'] = text
                                    org_found = True
                                    if debug:
                                        print(f"         ✓ 기관명 추출 (방법2): {text}")
                                    break
                            if org_found:
                                break
                        except:
                            continue
                except Exception as e:
                    if debug:
                        print(f"         방법2 실패: {e}")
            
            # 방법 3: 구조적 패턴으로 찾기
            if not org_found:
                # cl-control cl-container 내에서 찾기
                all_containers = soup.find_all('div', class_='cl-control cl-container')
                for container in all_containers:
                    # diamond-blt이 있는지 확인
                    diamond = container.find('div', class_='diamond-blt')
                    if diamond and '기관명' in clean_text(diamond.get_text()):
                        # 같은 컨테이너 내의 다른 cl-text 찾기
                        all_texts = container.find_all('div', class_='cl-text')
                        for text_div in all_texts:
                            text = clean_text(text_div.get_text())
                            if text and text != '기관명' and len(text) > 2:
                                basic_info['기관명'] = text
                                org_found = True
                                if debug:
                                    print(f"         ✓ 기관명 추출 (방법3): {text}")
                                break
                        if org_found:
                            break
            
            if not org_found and debug:
                print(f"         ⚠️ 기관명 추출 실패")
                
        except Exception as e:
            if debug:
                print(f"         기관명 추출 오류: {e}")
        
        # 표 정보 추출 (사업상태, 사업기간, 연락처, 이메일)
        try:
            # 사업상태
            status_xpath = "//div[contains(text(), '사업상태')]/../../following-sibling::*//div[contains(@class, 'cl-text')]"
            status_elem = driver.find_element(By.XPATH, status_xpath)
            basic_info['사업상태'] = clean_text(status_elem.text)
        except:
            pass
        
        try:
            # 사업기간
            period_elem = driver.find_element(By.CSS_SELECTOR, "div[aria-label='사업기간']")
            basic_info['사업기간'] = clean_text(period_elem.text)
        except:
            pass
        
        try:
            # 연락처와 이메일 추출
            table_cells = driver.find_elements(By.CSS_SELECTOR, "div.cl-control.cl-output")
            for i, cell in enumerate(table_cells):
                text = clean_text(cell.text)
                if text == '연락처' and i + 1 < len(table_cells):
                    basic_info['연락처'] = clean_text(table_cells[i + 1].text)
                elif text == '이메일' and i + 1 < len(table_cells):
                    basic_info['이메일'] = clean_text(table_cells[i + 1].text)
        except:
            pass
        
        # BeautifulSoup으로 재시도
        if '연락처' not in basic_info or not basic_info.get('연락처'):
            phone_pattern = r'(\d{2,3}[-\s]?\d{3,4}[-\s]?\d{4})'
            phones = re.findall(phone_pattern, soup.get_text())
            if phones:
                basic_info['연락처'] = phones[0]
        
        if '이메일' not in basic_info or not basic_info.get('이메일'):
            email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
            emails = re.findall(email_pattern, soup.get_text())
            if emails:
                basic_info['이메일'] = emails[0]
                
    except Exception as e:
        if debug:
            print(f"    ⚠️ 기본 정보 추출 오류: {e}")
    
    return basic_info


def extract_section_content(driver, section_title, all_section_contents, debug=False):
    """특정 섹션의 내용 추출 - 중복 방지 개선 버전"""
    content = ""
    
    try:
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # 방법 1: 정확한 구조로 섹션 찾기 (line-tit 다음의 lh-26)
        try:
            # 모든 line-tit 찾기
            line_tit_divs = soup.find_all('div', class_='line-tit')
            
            for line_tit_div in line_tit_divs:
                # 섹션 제목 확인
                title_elem = line_tit_div.find('div', class_='cl-text')
                if not title_elem:
                    continue
                    
                title_text = clean_text(title_elem.get_text())
                if title_text != section_title:
                    continue
                
                # 이 섹션의 부모 layout-wrap 찾기
                parent_wrap = line_tit_div.find_parent('div', class_='cl-layout-wrap')
                if not parent_wrap:
                    continue
                
                # 이 wrap 내에서 다음 layout-wrap 찾기 (제목 다음에 오는 내용)
                next_wrap = parent_wrap.find_next_sibling('div', class_='cl-layout-wrap')
                if next_wrap:
                    # lh-26 클래스 찾기
                    content_div = next_wrap.find('div', class_='lh-26')
                    if content_div:
                        text = clean_text(content_div.get_text())
                        
                        # 중복 체크: 다른 섹션에 이미 있는 내용인지 확인
                        is_duplicate = False
                        for other_section, other_content in all_section_contents.items():
                            if other_section != section_title and other_content and text in other_content:
                                is_duplicate = True
                                break
                        
                        if not is_duplicate and text and len(text) > 10:
                            content = text
                            if debug:
                                print(f"         ✓ 방법1 성공: {len(text)}자")
                            break
                
        except Exception as e:
            if debug:
                print(f"        방법1 오류: {e}")
        
        # 방법 2: XPath로 직접 찾기
        if not content:
            try:
                # uuid로 시작하는 id를 가진 layout-wrap 찾기
                all_wraps = soup.find_all('div', class_='cl-layout-wrap', id=True)
                
                for wrap in all_wraps:
                    # line-tit 찾기
                    line_tit = wrap.find('div', class_='line-tit')
                    if not line_tit:
                        continue
                    
                    title_text = clean_text(line_tit.get_text())
                    if title_text != section_title:
                        continue
                    
                    # 같은 wrap 내의 lh-26 찾기
                    content_div = wrap.find('div', class_='lh-26')
                    if content_div:
                        text = clean_text(content_div.get_text())
                        
                        # 중복 체크
                        is_duplicate = False
                        for other_section, other_content in all_section_contents.items():
                            if other_section != section_title and other_content and text in other_content:
                                is_duplicate = True
                                break
                        
                        if not is_duplicate and text and len(text) > 10:
                            content = text
                            if debug:
                                print(f"         ✓ 방법2 성공: {len(text)}자")
                            break
                            
            except Exception as e:
                if debug:
                    print(f"        방법2 오류: {e}")
        
        # 방법 3: 패턴 매칭으로 찾기
        if not content:
            try:
                # 섹션 제목을 포함하는 모든 요소 찾기
                all_containers = soup.find_all('div', class_='cl-control cl-container')
                
                for container in all_containers:
                    # line-tit이 있는지 확인
                    line_tit = container.find('div', class_='line-tit')
                    if not line_tit:
                        continue
                    
                    if clean_text(line_tit.get_text()) == section_title:
                        # 이 컨테이너 내의 lh-26 찾기
                        content_div = container.find('div', class_='lh-26')
                        if content_div:
                            text = clean_text(content_div.get_text())
                            
                            # 중복 체크
                            is_duplicate = False
                            for other_section, other_content in all_section_contents.items():
                                if other_section != section_title and other_content and text in other_content:
                                    is_duplicate = True
                                    break
                            
                            if not is_duplicate and text and len(text) > 10:
                                content = text
                                if debug:
                                    print(f"         ✓ 방법3 성공: {len(text)}자")
                                break
                                
            except Exception as e:
                if debug:
                    print(f"        방법3 오류: {e}")
                            
    except Exception as e:
        if debug:
            print(f"    ⚠️ '{section_title}' 섹션 추출 오류: {e}")
    
    return content


def extract_attachments(driver, debug=False):
    """첨부파일 정보 추출"""
    attachments = []
    
    try:
        # 첨부파일 섹션 찾기
        attachment_elements = driver.find_elements(By.CSS_SELECTOR, "div.blt-tit-m")
        
        for elem in attachment_elements:
            try:
                filename = clean_text(elem.text)
                if filename and len(filename) > 3:
                    attachments.append(filename)
            except:
                continue
        
        # BeautifulSoup으로 재시도
        if not attachments:
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            file_elements = soup.find_all('div', class_='blt-tit-m')
            for elem in file_elements:
                filename = clean_text(elem.get_text())
                if filename and len(filename) > 3:
                    attachments.append(filename)
                    
    except Exception as e:
        if debug:
            print(f"    ⚠️ 첨부파일 추출 오류: {e}")
    
    return '\n'.join(attachments) if attachments else ""


def extract_detail_info(driver, current_tab_id, debug=False):
    """상세 페이지에서 모든 정보 추출"""
    data = {
        '상세URL': driver.current_url, 
        '제목': '제목 없음', 
        '구분': TAB_SECTIONS.get(current_tab_id, '알 수 없음')
    } 
    
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.wlfare-info-nm div.cl-text"))
        )
        time.sleep(4) 
        
        wait_for_dynamic_content(driver, timeout=5)
        
        # 기본 정보 추출
        basic_info = extract_basic_info(driver, debug=debug)
        data.update(basic_info)
        
        # 각 섹션 내용 추출 - 섹션별 독립 추출
        sections = ['사업목적', '지원대상', '지원내용', '신청방법', '제출서류']
        all_section_contents = {}  # 중복 체크용
        
        for section in sections:
            if debug:
                print(f"      → '{section}' 섹션 처리 중...")
            
            content = extract_section_content(driver, section, all_section_contents, debug=debug)
            
            if content:
                data[section] = content
                all_section_contents[section] = content
                if debug:
                    print(f"         ✓ 내용 추출 완료 ({len(content)}자)")
                    print(f"         미리보기: {content[:100]}...")
            else:
                if debug:
                    print(f"         ⚠️ 내용 없음")
        
        # 첨부파일 추출
        if debug:
            print(f"      → '첨부파일' 섹션 처리 중...")
        
        attachments = extract_attachments(driver, debug=debug)
        if attachments:
            data['첨부파일'] = attachments
            if debug:
                print(f"         ✓ {len(attachments.split(chr(10)))}개 파일 추출")
        
    except Exception as e:
        if debug:
            print(f"      ✗ 상세 정보 추출 실패: {e}")
        pass
            
    return data

# ========================================
# 3. 메인 실행 블록
# ========================================

def save_data_to_files(data_list, filename_prefix, columns_order):
    """주어진 데이터를 DataFrame으로 만들어 CSV, TSV, JSON 파일로 저장"""
    if not data_list:
        print(f"⚠️ 저장할 데이터가 없습니다: {filename_prefix}")
        return []
        
    df = pd.DataFrame(data_list)
    existing_columns = [col for col in columns_order if col in df.columns]
    df = df.reindex(columns=existing_columns, fill_value='')
    
    saved_files = []

    # 1. TSV 저장
    tsv_filename = f'{filename_prefix}.tsv'
    df.to_csv(tsv_filename, index=False, sep='\t', encoding='utf-8-sig')
    print(f"\n🎉 {len(data_list)}개 항목을 '{tsv_filename}' (TSV)에 저장했습니다.")
    saved_files.append(tsv_filename)
    
    # 2. CSV 저장
    csv_filename = f'{filename_prefix}.csv'
    df.to_csv(csv_filename, index=False, sep=',', encoding='utf-8-sig')
    print(f"🎉 {len(data_list)}개 항목을 '{csv_filename}' (CSV)에 저장했습니다.")
    saved_files.append(csv_filename)

    # 3. JSON 저장 
    json_filename = f'{filename_prefix}.json'
    df.to_json(json_filename, orient='records', force_ascii=False, indent=4)
    print(f"🎉 {len(data_list)}개 항목을 '{json_filename}' (JSON)에 저장했습니다.")
    saved_files.append(json_filename)
    
    return saved_files

print("✅ 타겟사이트 청년 복지정보 민간 크롤링을 시작합니다.")
print("="*70)

# --- 설정 변수 ---
DEBUG_MODE = True  # 디버그 모드 (True로 설정하면 상세 로그 출력)
COLLECT_ALL_ITEMS = True  # ⭐ False: 테스트용 1개만 수집
TEST_MODE = False  # ⭐ 테스트 모드: 첫 번째 항목 1개만 크롤링

# 1. base_url
base_url = "https://www.target_site.go.kr/ssis-tbu/twataa/wlfareInfo/moveTWAT52005M.do" 

# 2. 크롤링할 탭 ID (3: 민간)
TAB_IDS_TO_CRAWL = [3]

# 3. 크롤링할 페이지 범위
START_PAGE = 1
END_PAGE = 11  # 테스트용 1페이지만

# 파일 저장에 필요한 컬럼 순서
COLUMNS_ORDER = ['순번', '페이지', '구분', '제목', '기관명', 
                 '사업상태', '사업기간', '연락처', '이메일',
                 '사업목적', '지원대상', '지원내용', '신청방법', 
                 '제출서류', '첨부파일', '상세URL']

# 드라이버 설정
chrome_options = Options()
chrome_options.add_argument('--headless')  # 헤드리스 모드
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')  # GPU 비활성화
chrome_options.add_argument('--lang=ko-KR')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option('prefs', {'intl.accept_languages': 'ko,ko-KR'})
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

print("\n⚙️ 설정:")
print(f"   - 디버그 모드: {DEBUG_MODE}")
print(f"   - 테스트 모드: {TEST_MODE} (1개 항목만 크롤링)")
print(f"   - 페이지 범위: {START_PAGE}~{END_PAGE}페이지")
print("="*70)

try:
    driver = webdriver.Chrome(options=chrome_options)
    driver.maximize_window()
    driver.execute_cdp_cmd('Network.setUserAgentOverride', {
        "userAgent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        "acceptLanguage": "ko-KR,ko;q=0.9"
    })
except Exception as e:
    print(f"❌ 드라이버 초기화 오류: {e}")
    sys.exit()

all_section_files = [] 

# --- 크롤링 루프 시작 ---
try:
    for tab_id in TAB_IDS_TO_CRAWL:
        section_name = TAB_SECTIONS.get(tab_id, f'tabId={tab_id}')
        
        section_data = []
        file_prefix = f'타겟사이트트_청년복지_{section_name}_{START_PAGE}~{END_PAGE}페이지_전체항목'

        print(f"\n\n{'#'*70}")
        print(f"🎯 섹션 크롤링 시작: {section_name} (tabId={tab_id})")
        print(f"   📊 크롤링 범위: {START_PAGE}~{END_PAGE}페이지")
        print(f"   📈 수집 모드: {'전체 항목' if COLLECT_ALL_ITEMS else '페이지당 9개'}")
        print(f"{'#'*70}")
        
        # 페이지 루프
        for page in range(START_PAGE, END_PAGE + 1):
            print(f"\n{'='*60}")
            print(f"📖 {section_name} - {page}/{END_PAGE} 페이지 크롤링 중...")
            print(f"{'='*60}")
             
            url = f"{base_url}?page={page}&orderBy=date&tabId={tab_id}&period=%EC%B2%AD%EB%85%84"
            driver.get(url)
            time.sleep(5)
             
            button_xpath = "//a[contains(@aria-label, '자세히 보기') or contains(text(), '자세히 보기')]"
             
            try:
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.XPATH, button_xpath))
                )
                buttons = driver.find_elements(By.XPATH, button_xpath)
            except:
                print(f"\n❌ '{section_name}' - '{page} 페이지'에서 버튼을 찾을 수 없습니다. 다음 페이지로 이동합니다.")
                continue
             
            # ⭐ 테스트 모드: 1개, 전체 모드: 모든 항목 또는 9개
            if TEST_MODE:
                items_to_process = 1
            elif COLLECT_ALL_ITEMS:
                items_to_process = len(buttons)
            else:
                items_to_process = min(9, len(buttons))
             
            print(f"\n총 {len(buttons)}개 항목 중 {items_to_process}개 처리...")
             
            for i in range(items_to_process):
                try:
                    # 매번 버튼 목록 새로 가져오기 (DOM 변경 대비)
                    current_buttons = driver.find_elements(By.XPATH, button_xpath)
                     
                    if i >= len(current_buttons): 
                        break
                     
                    button_to_click = current_buttons[i]
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button_to_click)
                    time.sleep(0.5)
                     
                    print(f"  ➡️ [{i+1}/{items_to_process}] 항목 클릭...")
                     
                    driver.execute_script("arguments[0].click();", button_to_click)
                    time.sleep(3)
                     
                    # 상세 정보 추출
                    detail_data = extract_detail_info(driver, tab_id, debug=DEBUG_MODE)
                     
                    if detail_data and detail_data.get('상세URL'):
                        detail_data['페이지'] = page
                        detail_data['순번'] = len(section_data) + 1
                        section_data.append(detail_data)
                         
                        # 수집된 필드 카운트
                        filled_fields = sum(1 for v in detail_data.values() if v and v != '제목 없음')
                        total_fields = len(COLUMNS_ORDER)
                         
                        print(f"     ✅ {detail_data['제목'][:50]}")
                        print(f"        총 {filled_fields}/{total_fields}개 필드 수집")
                        
                        # 각 섹션별 수집 상태 표시
                        print(f"        📋 섹션별 수집 결과:")
                        for field in ['사업목적', '지원대상', '지원내용', '신청방법', '제출서류', '첨부파일']:
                            if detail_data.get(field):
                                preview = detail_data[field][:30].replace('\n', ' ')
                                print(f"           ✓ {field}: {preview}...")
                            else:
                                print(f"           ✗ {field}: 없음")
                         
                        # 누락된 주요 필드 표시
                        important_fields = ['기관명', '사업목적', '지원대상', '지원내용', '신청방법', '제출서류']
                        missing = [f for f in important_fields if not detail_data.get(f)]
                        if missing:
                            print(f"        ⚠️ 누락: {', '.join(missing)}")
                    else:
                        print(f"     ❌ 정보 추출 실패")
                     
                    # 목록 페이지로 돌아가기
                    driver.back()
                    time.sleep(3)
                     
                except Exception as e:
                    print(f"     ⚠️ 오류 ({i+1}번째): {type(e).__name__}")
                    try:
                        driver.get(url) 
                        time.sleep(5)
                    except:
                        break 
                    continue
            
            # 페이지별 진행상황 출력
            print(f"\n📊 현재까지 수집된 항목: {len(section_data)}개")
             
        # 섹션별 저장
        if section_data:
            print(f"\n\n--- {section_name} 섹션 저장 시작 ---")
            print(f"📦 총 수집 항목: {len(section_data)}개")
            saved_files = save_data_to_files(section_data, file_prefix, COLUMNS_ORDER)
            all_section_files.extend(saved_files)
            print(f"--- {section_name} 저장 완료 ---\n")
        else:
            print(f"\n⚠️ {section_name} 섹션에서 수집된 데이터가 없습니다.")
                            
except Exception as e:
    print(f"\n❌ 크롤링 중 오류: {e}")
    if DEBUG_MODE:
        import traceback
        traceback.print_exc()
         
finally:
    try: 
        driver.quit()
    except: 
        pass

print("\n" + "="*70)
print("🎉 크롤링 완료!")
print(f"📊 총 {len(section_data) if 'section_data' in locals() else 0}개 항목 수집")
print(f"📁 저장된 파일: {len(all_section_files)}개")
for file in all_section_files:
    print(f"   - {file}")
print("="*70)